# Cryptocurrency Market Regime Detection

Identify calm, trending and stressed crypto regimes from rolling market behaviour.

**Portfolio category:** Financial clustering

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Generate chronological market regimes

In [ ]:
n = 1500
transition = np.array([[0.95, 0.04, 0.01], [0.05, 0.90, 0.05], [0.04, 0.10, 0.86]])
regimes = np.zeros(n, dtype=int)
for i in range(1, n):
    regimes[i] = rng.choice(3, p=transition[regimes[i - 1]])
means = np.array([0.0003, 0.0015, -0.0020])
volatilities = np.array([0.012, 0.025, 0.055])
returns = rng.normal(means[regimes], volatilities[regimes])
price = 25000 * np.exp(np.cumsum(returns))
volume = rng.lognormal(12 + regimes * 0.35, 0.45)
market = pd.DataFrame({
    "date": pd.date_range("2021-01-01", periods=n, freq="D"),
    "price": price,
    "return": returns,
    "volume": volume,
    "hidden_regime": regimes,
})
market["volatility_14d"] = market["return"].rolling(14).std()
market["momentum_14d"] = market["price"].pct_change(14)
market["volume_change"] = market["volume"].pct_change().clip(-5, 5)
market = market.dropna().reset_index(drop=True)
feature_names = ["return", "volatility_14d", "momentum_14d", "volume_change"]
display(market.head())

## 3. Market feature quality

In [ ]:
display(market[feature_names].describe().T)
print("Missing cells:", int(market[feature_names].isna().sum().sum()))

## 4. Scale rolling features

In [ ]:
X = StandardScaler().fit_transform(market[feature_names])

## 5. Select and fit market clusters

In [ ]:
rows = []
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE).fit_predict(X)
    rows.append({"k": k, "silhouette": silhouette_score(X, labels)})
scores = pd.DataFrame(rows)
best_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
market["cluster"] = KMeans(n_clusters=best_k, n_init=50, random_state=RANDOM_STATE).fit_predict(X)
display(scores.round(3))

## 6. Regime diagnostics

In [ ]:
display(pd.Series({
    "selected_k": best_k,
    "silhouette": silhouette_score(X, market["cluster"]),
    "adjusted_rand_vs_hidden_regime": adjusted_rand_score(market["hidden_regime"], market["cluster"]),
    "cluster_transitions": int(market["cluster"].diff().ne(0).sum() - 1),
}).to_frame("value"))

## 7. Cluster profiles

In [ ]:
profile = market.groupby("cluster")[feature_names].mean()
profile["days"] = market.groupby("cluster").size()
display(profile.round(4))

## 8. Chronological regime view

In [ ]:
sns.scatterplot(data=market, x="date", y="price", hue="cluster", palette="tab10", s=18)
plt.yscale("log")
plt.title("Detected crypto market regimes")
plt.tight_layout()

## 9. Key findings

Clusters describe recurring behaviour; they do not forecast returns and must not be treated as trading advice.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For cryptocurrency market regime detection,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.